<a href="https://colab.research.google.com/github/sinskid/deep_learning_project/blob/main/notebooks/models_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check if we're running in Colab and set up the environment accordingly
import os
IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    !git clone https://github.com/sinskid/deep_learning_project.git
    %cd deep_learning_project
    !pip install -r requirements.txt

import sys
repo_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_path not in sys.path:
    sys.path.append(repo_path)
from python.data import load_testdata, load_traindata
from python.models import get_cnn , get_vit
from python.utils import setup_dirs
from python.train import train
from torch import nn
from torch import optim
import torch

Cloning into 'deep_learning_project'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 53 (delta 18), reused 35 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 23.02 KiB | 3.84 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/deep_learning_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 8.0 MB/s eta 0:00:00


**SET-UP FICHIERS**

In [2]:
# Set up directories for outputs, models, and logs
paths = setup_dirs()

**SET-UP MODELS AND DEVICE**

In [3]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
# Initialize the CNN model and move it to the appropriate device
model_cnn = get_cnn().to(device)

# Initialize the ViT model and move it to the appropriate device
model_vit = get_vit().to(device)

cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 128MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

**ENTRAINEMENT CNN**

In [4]:
# Dataloaders cnn
cnn_dataloader_train = load_traindata(batch_size=32,num_workers=2, model_name = "cnn")
cnn_dataloader_test = load_testdata(batch_size=32,num_workers=2, model_name = "cnn")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5400 [00:00<?, ? examples/s]

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.00001)

# NB_EPOCHS Set to 10 for Colab, 1 for local testing
if device == "cuda":
    NB_EPOCHS = 20
else:
    NB_EPOCHS = 1

# Check if a saved model already exists
if os.path.exists(os.path.join(paths['models'], 'cnn_model.pth')):
    print("Cnn Model exists")

# Train the model with early stopping if no model is found in the models directory
else:
    valid_loss = float('inf')
    for epoch in range(NB_EPOCHS):
        train_loss, test_loss = train(model_cnn, cnn_dataloader_train, cnn_dataloader_test, optimizer, criterion, device)
        print(f"Epoch {epoch+1}/{NB_EPOCHS}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
        # Early stopping condition
        if test_loss < valid_loss:
            valid_loss = test_loss
            torch.save(model_cnn.state_dict(), os.path.join(paths['models'], 'cnn_model.pth'))
        else:
            print("Early stopping triggered.")
            break

Epoch 1/20, Train Loss: 0.6227, Test Loss: 0.1507
Epoch 2/20, Train Loss: 0.1640, Test Loss: 0.0903
Epoch 3/20, Train Loss: 0.1104, Test Loss: 0.0798
Epoch 4/20, Train Loss: 0.0745, Test Loss: 0.0694
Epoch 5/20, Train Loss: 0.0564, Test Loss: 0.0680
Epoch 6/20, Train Loss: 0.0440, Test Loss: 0.0648
Epoch 7/20, Train Loss: 0.0333, Test Loss: 0.0574
Epoch 8/20, Train Loss: 0.0255, Test Loss: 0.0632
Early stopping triggered.


**ENTRAINEMENT ViT**

In [6]:
# Dataloaders vit
vit_dataloader_train = load_traindata(batch_size=32,num_workers=2, model_name = "vit")
vit_dataloader_test = load_testdata(batch_size=32,num_workers=2, model_name = "vit")

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_vit.parameters(), lr=0.00001)

# NB_EPOCHS Set to 10 for Colab, 1 for local testing
if device == "cuda":
    NB_EPOCHS = 10
else:
    NB_EPOCHS = 1

# Check if a saved model already exists
model_path = os.path.join(paths['models'], 'vit_model.pth')
if os.path.exists(model_path):
    print("ViT Model exists")

# Train the model with early stopping if no model is found in the models directory
else:
    valid_loss = float('inf')
    for epoch in range(NB_EPOCHS):
        train_loss, test_loss = train(model_vit, vit_dataloader_train, vit_dataloader_test, optimizer, criterion, device)
        print(f"Epoch {epoch+1}/{NB_EPOCHS}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
        # Early stopping condition
        if test_loss < valid_loss:
            valid_loss = test_loss
            torch.save(model_vit.state_dict(), os.path.join(paths['models'], 'vit_model.pth'))
        else:
            print("Early stopping triggered.")
            break


Epoch 1/10, Train Loss: 0.3337, Test Loss: 0.1031
Epoch 2/10, Train Loss: 0.0617, Test Loss: 0.0731
Epoch 3/10, Train Loss: 0.0283, Test Loss: 0.0804
Early stopping triggered.
